In [1]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
# res='1km';t_res='5min'
# Np_str='1e6'

# # dx = 1km; Np = 50M
# #Importing Model Data
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# dx = 1km; Np = 50M; Nz = 95
#Importing Model Data
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min_95nz.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_95nz.nc') #***
res='1km'; t_res='1min_95nz'; Np_str='50e6'

In [2]:
times=data['time'].values/(1e9 * 60); times=times.astype(float);
minutes=1/times[1] #1 / minutes per timestep = timesteps per minute
index_adjust=0
ocean_fraction=0.25

In [3]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

#####

#Import StatisticalFunctions 
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import StatisticalFunctions
from StatisticalFunctions import * # import NumericalFunctions 

In [4]:
#NEEDED TO PLOT THE CORRECT DATA #*#*
data_type="Tracked_Properties"

In [5]:
def limit_axes_to_y(ax, y_min=0, y_max=7, buffer_frac=0.1):
    ax.set_ylim(y_min, y_max)

    x_limited = []
    for line in ax.get_lines():
        xdata, ydata = np.array(line.get_xdata()), np.array(line.get_ydata())
        # Mask for y within limits
        y_mask = (ydata >= y_min) & (ydata <= y_max)
        # Apply mask
        x_visible = xdata[y_mask]
        # Remove NaN or Inf from x_visible
        x_visible = x_visible[np.isfinite(x_visible)]
        x_limited.extend(x_visible)
    if len(x_limited) > 0:
        x_limited = np.array(x_limited)
        x_min, x_max = np.min(x_limited), np.max(x_limited)

        if not (np.isfinite(x_min) and np.isfinite(x_max)):
            print("Warning: Non-finite x-limits detected, skipping set_xlim")
            return
        x_range = x_max - x_min

        if x_range == 0:
            buffer = 0.1  # fixed small buffer
        else:
            buffer = buffer_frac * x_range
        ax.set_xlim(x_min - buffer, x_max + buffer)
    else:
        print("Warning: No visible x data within y limits to set xlim")


In [6]:
######################################
#PLOTTING

In [7]:
#LIMITING Y AXIS
limit_y=True
limit_y=False

In [8]:
def LoadAllCloudBase():
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
    in_file = dir2 + f"all_cloudbase_{res}_{t_res}_{Np_str}.pkl"
    with open(in_file, 'rb') as f:
        all_cloudbase = pickle.load(f)
    return(all_cloudbase)
min_all_cloudbase=np.nanmin(LoadAllCloudBase())
all_cloudbase=min_all_cloudbase
print(f"Minimum Cloudbase is: {min_all_cloudbase}\n")

Minimum Cloudbase is: 1.225000023841858



In [9]:
def LoadMeanLFC():
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
    in_file = dir2 + f"MeanLFC_{res}_{t_res}_{Np_str}.pkl"
    with open(in_file, 'rb') as f:
        MeanLFC = pickle.load(f)
    return MeanLFC
MeanLFC=LoadMeanLFC()
print(f"Mean LFC is: {MeanLFC}\n")

Mean LFC is: 1856.2518881791163



In [10]:
def averaged_profiles(profile):
    out_var=profile[ (profile[:, 1] > 1)]; #gets rid of rows that have no data\n"
    out_var=np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T #divides the data column by the counter column
    return out_var

In [11]:
# List of variables with their corresponding labels and x-axis titles
var_units = [
    ("w", 'w (m/s)'),
    ("qv", r"$q_v$ (g/kg)"),
    ("qcqi", r"$q_c+q_i$ (g/kg)"),
    ("th", r"$\theta$ (K)"),
    ("th_e", r"$\theta_e$ (K)"),
    ("buoyancy", r"Buoyancy $(m/s^2)$"),
    ("hmc", "HMC (g/kg/s)"),
]

In [14]:
type1='CL';type2='nonCL'
dir3=dir+'Project_Algorithms/Tracked_Profiles/OUTPUT_FILES/'
filePath=dir3+f"{data_type}_"+f"{type1}_{type2}_tracked_profiles_{res}_{t_res}_{Np_str}.h5"
key_list=[]
with h5py.File(filePath, 'r') as h5f:
    for key in h5f.keys():
        globals()[key] = h5f[key][:]
        if '_squares' not in key:
            key_list.append(key)

# #CALCULATING STANDARD DEVIATION
# for key in key_list:
#     # globals()[key+f"_SE"]=ProfileStandardError(globals()[key],globals()[key+f"_squares"])
#     globals()[key+f"_SE"]=ProfileStandardDeviation(globals()[key],globals()[key+f"_squares"])
#     # print(key)

In [ ]:
def load_and_prepare(type1, type2):
    th = globals()[f"{type1}_{type2}_profile_array_TH"]  # e.g. type1:CL type2:ALL
    th_e = globals()[f"{type1}_{type2}_profile_array_TH_E"]  # e.g. type1:CL type2:ALL

    def prep(p):
        ratio = np.empty_like(p[:, 0])
        np.divide(p[:, 0], p[:, 1], out=ratio, where=p[:, 1] != 0)
        ratio[p[:, 1] == 0] = np.nan  # explicitly set where denom=0
        return np.column_stack([ratio, p[:, 2]])

    return prep(th), prep(th_e)

def plot_profiles_and_derivs(p_th_all, p_th_e_all,
                            p_th_shallow, p_th_e_shallow,
                            p_th_deep, p_th_e_deep,
                            out_all, out_e_all,
                            out_shallow, out_e_shallow,
                            out_deep, out_e_deep,
                            cloudbase):

    colors = {'ALL': 'k', 'SHALLOW': 'g', 'DEEP': 'b'}
    linestyles = {'theta': '--', 'theta_e': '-'}
    lw_thin = 0.8  # line width for thinner lines
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Left subplot: profiles
    ax = axes[0]
    ax.plot(p_th_all[:, 0], p_th_all[:, 1], color=colors['ALL'], linestyle=linestyles['theta'], label='theta (ALL)', linewidth=lw_thin)
    ax.plot(p_th_e_all[:, 0], p_th_e_all[:, 1], color=colors['ALL'], linestyle=linestyles['theta_e'], label='theta_e (ALL)', linewidth=lw_thin)
    ax.plot(p_th_shallow[:, 0], p_th_shallow[:, 1], color=colors['SHALLOW'], linestyle=linestyles['theta'], label='theta (SHALLOW)', linewidth=lw_thin)
    ax.plot(p_th_e_shallow[:, 0], p_th_e_shallow[:, 1], color=colors['SHALLOW'], linestyle=linestyles['theta_e'], label='theta_e (SHALLOW)', linewidth=lw_thin)
    ax.plot(p_th_deep[:, 0], p_th_deep[:, 1], color=colors['DEEP'], linestyle=linestyles['theta'], label='theta (DEEP)', linewidth=lw_thin)
    ax.plot(p_th_e_deep[:, 0], p_th_e_deep[:, 1], color=colors['DEEP'], linestyle=linestyles['theta_e'], label='theta_e (DEEP)', linewidth=lw_thin)
    ax.axhline(cloudbase, color='purple', linestyle='dashed')
    ax.axhline(MeanLFC/1000,color='green',linestyle='dashed')

    ax.set_ylim(bottom=0)
    if limit_y==True: ax.set_ylim(top=7)
    ax.set_xlabel('(K)')
    ax.set_ylabel('z (km)')
    ax.legend(fontsize='small')
    ax.set_title('Profiles')

    # Right subplot: derivatives
    ax = axes[1]
    ax.plot(out_all[:, 0], out_all[:, 1], color=colors['ALL'], linestyle=linestyles['theta'], label='d/dz(theta) (ALL)', linewidth=lw_thin)
    # ax.plot(out_e_all[:, 0], out_e_all[:, 1], color=colors['ALL'], linestyle=linestyles['theta_e'], label='d/dz(theta_e) (ALL)', linewidth=lw_thin)
    ax.plot(out_shallow[:, 0], out_shallow[:, 1], color=colors['SHALLOW'], linestyle=linestyles['theta'], label='d/dz(theta) (SHALLOW)', linewidth=lw_thin)
    # ax.plot(out_e_shallow[:, 0], out_e_shallow[:, 1], color=colors['SHALLOW'], linestyle=linestyles['theta_e'], label='d/dz(theta_e) (SHALLOW)', linewidth=lw_thin)
    ax.plot(out_deep[:, 0], out_deep[:, 1], color=colors['DEEP'], linestyle=linestyles['theta'], label='d/dz(theta) (DEEP)', linewidth=lw_thin)
    # ax.plot(out_e_deep[:, 0], out_e_deep[:, 1], color=colors['DEEP'], linestyle=linestyles['theta_e'], label='d/dz(theta_e) (DEEP)', linewidth=lw_thin)
    ax.axvline(0, color='grey', linestyle='dashed', alpha=0.5)
    ax.axhline(cloudbase, color='purple', linestyle='dashed')
    ax.axhline(MeanLFC/1000,color='green',linestyle='dashed')

    ax.set_ylim(bottom=0)
    if limit_y==True: ax.set_ylim(top=7)
    ax.set_xlabel('(K)')
    ax.set_ylabel('z (km)')
    ax.legend(fontsize='small')
    ax.set_title('Vertical Derivatives')

    # Right subplot: derivatives
    ax = axes[2]
    # ax.plot(out_all[:, 0], out_all[:, 1], color=colors['ALL'], linestyle=linestyles['theta'], label='d/dz(theta) (ALL)', linewidth=lw_thin)
    ax.plot(out_e_all[:, 0], out_e_all[:, 1], color=colors['ALL'], linestyle=linestyles['theta_e'], label='d/dz(theta_e) (ALL)', linewidth=lw_thin)
    # ax.plot(out_shallow[:, 0], out_shallow[:, 1], color=colors['SHALLOW'], linestyle=linestyles['theta'], label='d/dz(theta) (SHALLOW)', linewidth=lw_thin)
    ax.plot(out_e_shallow[:, 0], out_e_shallow[:, 1], color=colors['SHALLOW'], linestyle=linestyles['theta_e'], label='d/dz(theta_e) (SHALLOW)', linewidth=lw_thin)
    # ax.plot(out_deep[:, 0], out_deep[:, 1], color=colors['DEEP'], linestyle=linestyles['theta'], label='d/dz(theta) (DEEP)', linewidth=lw_thin)
    ax.plot(out_e_deep[:, 0], out_e_deep[:, 1], color=colors['DEEP'], linestyle=linestyles['theta_e'], label='d/dz(theta_e) (DEEP)', linewidth=lw_thin)
    ax.axvline(0, color='grey', linestyle='dashed', alpha=0.5)
    ax.axhline(cloudbase, color='purple', linestyle='dashed')
    ax.axhline(MeanLFC/1000,color='green',linestyle='dashed')

    ax.set_ylim(bottom=0)
    ax.set_xlabel('(K)')
    ax.set_ylabel('z (km)')
    ax.legend(fontsize='small')
    ax.set_title('Vertical Derivatives')    

    plt.tight_layout()
    
    #LIMITING YAXIS TO BELOW 7 KM
    if limit_y==True: 
        # #LIMITING YAXIS TO BELOW 7 KM
        limit_axes_to_y(ax,y_min=0, y_max=7)


# === Main code ===

# Load and prepare profiles for ALL, SHALLOW, DEEP (general)
ALL_TH, ALL_TH_E = load_and_prepare('CL', 'ALL')
SHALLOW_TH, SHALLOW_TH_E = load_and_prepare('CL', 'SHALLOW')
DEEP_TH, DEEP_TH_E = load_and_prepare('CL', 'DEEP')

# Calculate derivatives
out_all = Profile_Ddz(ALL_TH)
out_e_all = Profile_Ddz(ALL_TH_E)
out_shallow = Profile_Ddz(SHALLOW_TH)
out_e_shallow = Profile_Ddz(SHALLOW_TH_E)
out_deep = Profile_Ddz(DEEP_TH)
out_e_deep = Profile_Ddz(DEEP_TH_E)

# Plot all together
plot_profiles_and_derivs(
    ALL_TH, ALL_TH_E,
    SHALLOW_TH, SHALLOW_TH_E,
    DEEP_TH, DEEP_TH_E,
    out_all, out_e_all,
    out_shallow, out_e_shallow,
    out_deep, out_e_deep,
    all_cloudbase
)
